# Tips 10：ファイルを種類別に整理

**業務の困りごと**：PDF・Excel・画像が混ざった作業フォルダを分類する。

**このTipsでできること**：`Path.suffix` で拡張子を取り出し、分類先のフォルダを作って `shutil.move` で移動します。定義のない拡張子は「その他」に入れます。

**使用ライブラリ**：pathlib, shutil

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

Excel・PDF・画像・テキストの拡張子を持つ練習用ファイルを作ります。

最後の `head()` で先頭の行を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
from pathlib import Path
import shutil, tempfile
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 毎回新しい作業用フォルダを作る
folder=Path(tempfile.mkdtemp(prefix='tips10_',dir=base))
# 種類が違う練習用ファイルを用意する
for name in ['売上.xlsx','資料.pdf','写真.jpg','メモ.txt']:
    (folder/name).write_text('分類練習用',encoding='utf-8')
import pandas as pd
print('整理前のファイル一覧（先頭5件）:')
pd.DataFrame({'ファイル名':[p.name for p in sorted(folder.iterdir()) if p.is_file()]}).head()


## 2. 自動処理

`Path.suffix` で拡張子を取り出し、分類先のフォルダを作って `shutil.move` で移動します。定義のない拡張子は「その他」に入れます。

**結果を見るポイント**：4ファイルが4つの分類先に移動し、結果をZIPで保存できます。


In [ ]:
# 拡張子と移動先の対応表を作る
groups={'.xlsx':'Excel','.pdf':'PDF','.jpg':'画像','.png':'画像'}
# 作業フォルダ直下のファイルを1つずつ見る
for src in list(folder.iterdir()):
    if not src.is_file(): continue
    # 拡張子から分類先を決める
    destination=folder/groups.get(src.suffix.lower(),'その他')
    destination.mkdir(exist_ok=True)
    target=destination/src.name
    if target.exists(): raise FileExistsError(target)
    # 分類先へ実際に移動する
    shutil.move(str(src),str(target))
print([(p.parent.name,p.name) for p in folder.glob('*/*')])
# 複数の成果物をPCでまとめて扱えるようZIPにする
import shutil
out = base / 'Tips10_成果物.zip'
shutil.make_archive(str(out.with_suffix('')), 'zip', root_dir=folder)
print('ダウンロード用ZIP:', out)


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

4ファイルが4つの分類先に移動し、結果をZIPで保存できます。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert len(list(folder.glob('*/*')))==4
print('確認OK')
assert out.exists()  # ダウンロード用ZIPも確認する


## 4. 持ち帰り・練習

**練習問題**：CSVファイル用の分類先も作りましょう。

実務では移動前に対象と保存先の一覧を確認し、バックアップを取ります。

**実務で使う前に**：練習用ファイルの中身は各形式の本物ではありません。実務のファイル移動前にはバックアップを取ってください。

複数のファイルをZIPでPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
